In [20]:
#LLM application (dummy)
import random

from IPython.core.debugger import prompt


class NakliLLM:
    def __init__(self):
        print('LLM created succesfully')

    def predict(self,prompt):
        response_list=[
            'Delhi is the capital of India',
            'Cell is the smallest unit of life',
            'Life is a journey',
            'Machine learning and Deep learning algorithms are becoming famous'
        ]
        return {'response':random.choice(response_list)}

In [21]:
llm=NakliLLM()
llm.predict('What is cell?')

{'response': 'Life is a journey'}

In [22]:
#dummy prompt_template class
class NakliPromptTemplate:
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def format(self,input_dict):
        return self.template.format(**input_dict)

In [23]:
#without using the concept of chains
template=NakliPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables={'length','topic'}
)
template.format({'length':'short','topic': 'India'})

'Write a short poem about India'

In [24]:
#dummy chain class
class NakliLLMChain:
    def __init__(self,llm,prompt):
        self.llm = llm
        self.prompt = prompt

    def run(self,input_dict):
       final_prompt=self.prompt.format(input_dict)
       result=self.llm.predict(final_prompt)
       return result['response']

In [25]:
#usig the concept of chains
template=NakliPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables={'length','topic'}
)

llm=NakliLLM()

chain= NakliLLMChain(llm,template)

chain.run({'length':'short','topic': 'India'})

'Life is a journey'

In [26]:
#dummy runnable class
from abc import ABC, abstractmethod

class Runnable(ABC):
    @abstractmethod
    def invoke(self,input_data):
        pass


In [27]:
#in this we are inheriting the runnable class so thet the llm class is its child with the invoke function available. Not removing the predict so that its still functional for those who are already using it

import random

class NakliLLM (Runnable):
    def __init__(self):
        print('LLM created succesfully')

    def invoke(self,prompt):
        response_list=[
            'Delhi is the capital of India',
            'Cell is the smallest unit of life',
            'Life is a journey',
            'Machine learning and Deep learning algorithms are becoming famous'
        ]
        return {'response':random.choice(response_list)}

    def predict(self,prompt):
        response_list=[
            'Delhi is the capital of India',
            'Cell is the smallest unit of life',
            'Life is a journey',
            'Machine learning and Deep learning algorithms are becoming famous'
        ]
        return {'response':random.choice(response_list)}

In [28]:
#the same inheritance applies to this class also
class NakliPromptTemplate(Runnable):
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def invoke(self,input_dict):
        return self.template.format(**input_dict)

    def format(self,input_dict):
        return self.template.format(**input_dict)

In [29]:
#dummy string output parser
class NakliStrOutputParser(Runnable):
    def __init__(self):
        pass
    def invoke(self,input_data):
        return input_data['response']


In [30]:
#dummy class to connect the runnables

class RunnableConnector(Runnable):
    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
        return input_data



In [31]:
parser=NakliStrOutputParser()

In [32]:
llm=NakliLLM()
template = NakliPromptTemplate(
    template="Write a {length} note about {topic}",
    input_variables=["length", "topic"]
)
chain= RunnableConnector([template,llm,parser])
chain.invoke({'length':'short','topic': 'India'})

'Life is a journey'

In [33]:
#joining two chains to get a bigger chain
template1=NakliPromptTemplate(
    template="Write a joke about {topic}",
    input_variables=["topic"]
)

In [34]:
template2=NakliPromptTemplate(
    template="Explain the following {response}",
    input_variables=["response"]
)

In [35]:
llm=NakliLLM()
parser=NakliStrOutputParser()
chain1=RunnableConnector([template1,llm])
chain2=RunnableConnector([template2,llm,parser])
final_chain=RunnableConnector([chain1,chain2])
final_chain.invoke({'topic': 'India'})

'Cell is the smallest unit of life'